# Investigate FFC sample size

How many FOVs (or frames) does flat-field correction (FFC) actually need to
converge to a stable estimate? Built entirely on the FFC functions in
`MERci.analysis.ffc` and `MERci.acquisition.positions.find_exterior_fovs` --
this notebook only calls and compares them, no new production logic lives
here. See `analysis/ffc.py`'s module docstring for the three FOV/frame
selection strategies this notebook compares.

Follows [`NOTEBOOK_GUIDELINES.md`](../../NOTEBOOK_GUIDELINES.md) (repo root):
calculation cells cached under `analysis/cache/investigate_ffc_sample_size/`,
progress reporting on the convergence sweeps, explicit plot font sizes.

**Question this notebook answers**: does FFC accuracy keep improving with
more FOVs, or does it converge after just a handful -- and does a single
near-empty FOV's full z-stack (many frames, one FOV) get there just as well
as many partially-filled exterior FOVs (one frame each)? The answer
determines `ExperimentConfig.ffc_fov_selection_strategy` and
`ffc_min_samples`'s production defaults (`common/config.py`) -- a manual
follow-up edit once this notebook has run against real data, not automated
by the notebook itself.

**Method**:
1. Build a **reference field** from the largest practical sample (every
   grid-exterior FOV of a chosen round/color, one mid-z frame each).
2. **Sweep A**: increasing N of the emptiest-by-stats FOVs (one frame each),
   compare each resulting field to the reference (percent difference).
3. **Sweep B**: increasing number of z-frames from a single near-empty FOV,
   same comparison.
4. Plot both convergence curves; report the smallest N/frame-count at which
   the difference drops below a tolerance.

## 1 -- Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress               import ProgressTracker
from MERci.progress_display       import ProgressReporter
from MERci.scheduler              import resolve_round_color_frame_indices
from MERci.analysis.ffc import (
    select_ffc_exterior_fovs, select_emptiest_fovs, select_all_frames_of_fov,
    compute_ffc_field_for_color,
)
from MERci.acquisition.configs import find_frame_table_for_hal_config

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 -- Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# Which round/color to investigate -- None = auto-pick the first
# fully-imaged round with a resolvable color (same convention
# analysis/ffc.py's resolve_ffc_reference_round uses in production).
ROUND_ID = None
COLOR_NM = None

# How many exterior FOVs to use for the reference field (the "ground truth"
# every sweep step is compared against) -- None = use every exterior FOV found.
REFERENCE_N_FOVS = None

# Sweep A: candidate FOV counts to test (the emptiest-by-stats strategy).
SWEEP_A_N_FOVS = [1, 2, 3, 5, 8, 12, 20]

# Sweep B: candidate frame counts to test, from a single near-empty FOV's
# own z-stack (the single_fov_all_frames strategy).
SWEEP_B_N_FRAMES = [1, 3, 5, 10, 20]

# Convergence tolerance (percent) -- the smallest N/frame-count at which the
# median percent difference vs. the reference field drops below this is
# reported as "enough".
CONVERGENCE_TOLERANCE_PCT = 2.0

# FFC field computation parameters -- same defaults as ExperimentConfig's
# ffc_* fields (common/config.py), kept explicit here so this investigation
# doesn't silently drift from whatever the production defaults become.
FFC_SMOOTH_SIGMA_PX      = 50.0
FFC_NORMALIZE_PERCENTILE = 99.99
FFC_MIN_VALUE            = 0.10
FFC_CONNECTIVITY         = "8"
FFC_NEIGHBOR_TOLERANCE   = 0.25

# Explicit plot font sizes (NOTEBOOK_GUIDELINES.md #5).
PLOT_TITLE_FONTSIZE  = 14
PLOT_LABEL_FONTSIZE  = 12
PLOT_TICK_FONTSIZE   = 11
PLOT_LEGEND_FONTSIZE = 10

print(f"Sample name   : {SAMPLE_NAME}")
print(f"Positions tag : {POSITIONS_TAG}")

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    ffc_connectivity        = FFC_CONNECTIVITY,
    ffc_neighbor_tolerance  = FFC_NEIGHBOR_TOLERANCE,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

NOTEBOOK_NAME = "investigate_ffc_sample_size"
cache_dir     = config.analysis_dir / "cache" / NOTEBOOK_NAME
cache_dir.mkdir(parents=True, exist_ok=True)

# Auto-pick the first fully-imaged round with a resolvable color, unless overridden above.
if ROUND_ID is None or COLOR_NM is None:
    for rid in meta.valid_round_ids():
        if not tracker.all_fovs_done_for_round(rid, meta, config.fov_subset):
            continue
        color_indices = resolve_round_color_frame_indices(rid, config, meta)
        if color_indices:
            ROUND_ID = ROUND_ID or rid
            COLOR_NM = COLOR_NM or next(iter(color_indices))
            break

if ROUND_ID is None or COLOR_NM is None:
    raise RuntimeError("No fully-imaged round with a resolvable color found -- "
                        "set ROUND_ID/COLOR_NM explicitly.")

color_indices = resolve_round_color_frame_indices(ROUND_ID, config, meta)
FRAME_IDX = color_indices[COLOR_NM]

print(f"Rounds       : {meta.n_rounds}")
print(f"FOVs         : {meta.n_fovs}")
print(f"ROUND_ID     : {ROUND_ID}")
print(f"COLOR_NM     : {COLOR_NM}")
print(f"FRAME_IDX    : {FRAME_IDX}")
print(f"Cache        : {cache_dir}")

## 3 -- Rank FOVs by existing stats (emptiest candidates)

Every FOV's per-frame stats are already computed by `analyze_file`/
`measure_stats` and sit in `analysis/stats/{stem}_stats.csv` -- ranking by
`(mean, std)` needs no new raw reads.

In [ ]:
round_info = meta.rounds[ROUND_ID]

ranked = []
for fov_id, file_list in round_info.fov_files.items():
    if not file_list:
        continue
    fpath = file_list[0]
    stats_path = tracker.stats_path(fpath)
    if not stats_path.exists():
        continue
    stats_df = pd.read_csv(stats_path)
    row = stats_df[stats_df["frame"] == FRAME_IDX]
    if row.empty:
        continue
    ranked.append({"fov_id": fov_id, "path": fpath,
                    "mean": float(row["mean"].iloc[0]), "std": float(row["std"].iloc[0])})

ranked_df = pd.DataFrame(ranked).sort_values(["mean", "std"]).reset_index(drop=True)
ranked_df.to_csv(cache_dir / "ranked_fovs.csv", index=False)
print(f"{len(ranked_df)} FOV(s) ranked; {ranked_df['path'].iloc[0].name if len(ranked_df) else 'n/a'} "
      f"is the emptiest candidate.")

Display the ranked table (emptiest first):

In [ ]:
ranked_df.head(20)

## 4 -- Reference field (largest practical sample)

Built from every grid-exterior FOV of `ROUND_ID`/`COLOR_NM`, one mid-z frame
each -- the closest proxy to "use every available exterior FOV" this
notebook can build, and what every sweep step below is compared against.

In [ ]:
reference_cache = cache_dir / "reference_field.npz"

if reference_cache.exists():
    _npz = np.load(reference_cache)
    reference_field = _npz["field"]
    n_reference_samples = int(_npz["n_samples"])
    print(f"Loaded cached reference field ({n_reference_samples} samples): {reference_cache}")
else:
    exterior_samples = select_ffc_exterior_fovs(ROUND_ID, config, meta, FRAME_IDX)
    if REFERENCE_N_FOVS is not None:
        exterior_samples = exterior_samples[:REFERENCE_N_FOVS]
    if len(exterior_samples) < 2:
        raise RuntimeError(f"Only {len(exterior_samples)} exterior FOV(s) found -- "
                            f"not enough to build a reference field.")

    reference_field, _meta = compute_ffc_field_for_color(
        exterior_samples, config.frame_width, config.frame_height,
        FFC_SMOOTH_SIGMA_PX, FFC_NORMALIZE_PERCENTILE, FFC_MIN_VALUE,
    )
    n_reference_samples = len(exterior_samples)
    np.savez_compressed(reference_cache, field=reference_field, n_samples=n_reference_samples)
    print(f"Reference field computed from {n_reference_samples} exterior FOV(s), cached: {reference_cache}")

Display the reference field:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
img = ax.imshow(reference_field, cmap="turbo")
ax.set_title(f"Reference FFC field ({n_reference_samples} exterior FOVs, {COLOR_NM}nm)",
             fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.colorbar(img, ax=ax)
fig.tight_layout()
fig.savefig(cache_dir / "reference_field.png", dpi=150)
plt.show()

## 5 -- Convergence sweep A: N emptiest FOVs (one frame each)

For each `N` in `SWEEP_A_N_FOVS`, build a field from the N emptiest-by-stats
FOVs and compare it to the reference field.

In [ ]:
def percent_diff_vs_reference(field, reference):
    valid = reference > 0
    diff = 100 * np.abs(field[valid] - reference[valid]) / reference[valid]
    return float(np.median(diff)), float(np.percentile(diff, 95))

sweep_a_cache = cache_dir / "sweep_a.csv"

if sweep_a_cache.exists():
    sweep_a_df = pd.read_csv(sweep_a_cache)
    print(f"Loaded cached sweep A results: {sweep_a_cache}")
else:
    rows = []
    reporter = ProgressReporter(total=len(SWEEP_A_N_FOVS), label="Sweep A (emptiest FOVs)")
    for n in reporter.wrap(SWEEP_A_N_FOVS):
        samples = select_emptiest_fovs(ROUND_ID, config, meta, tracker, FRAME_IDX, n)
        if len(samples) < n:
            continue
        field, _meta = compute_ffc_field_for_color(
            samples, config.frame_width, config.frame_height,
            FFC_SMOOTH_SIGMA_PX, FFC_NORMALIZE_PERCENTILE, FFC_MIN_VALUE,
        )
        median_pct, p95_pct = percent_diff_vs_reference(field, reference_field)
        rows.append({"n_fovs": n, "median_pct_diff": median_pct, "p95_pct_diff": p95_pct})

    sweep_a_df = pd.DataFrame(rows)
    sweep_a_df.to_csv(sweep_a_cache, index=False)

sweep_a_df

## 6 -- Convergence sweep B: one FOV, increasing number of frames

Using the single emptiest FOV from section 3's ranking, treat an increasing
prefix of its own z-frames (of `COLOR_NM`) as independent FFC samples.

In [ ]:
sweep_b_cache = cache_dir / "sweep_b.csv"

if sweep_b_cache.exists():
    sweep_b_df = pd.read_csv(sweep_b_cache)
    print(f"Loaded cached sweep B results: {sweep_b_cache}")
else:
    # Locate the round's frame table (same HAL-config -> frame-table resolution
    # scheduler.resolve_round_color_frame_indices uses internally).
    frame_table = None
    for s in meta.series_for_round(ROUND_ID):
        if s.hal_config:
            ft_path = find_frame_table_for_hal_config(config.settings_dir / s.hal_config, config.metadata_dir)
            if ft_path is not None:
                frame_table = pd.read_csv(ft_path, index_col=0)
                break
    if frame_table is None:
        raise RuntimeError(f"Could not resolve a frame table for round {ROUND_ID}.")

    emptiest_fov_path = ranked_df["path"].iloc[0]
    all_frame_samples = select_all_frames_of_fov(emptiest_fov_path, frame_table, COLOR_NM)
    print(f"{emptiest_fov_path.name} has {len(all_frame_samples)} frame(s) of {COLOR_NM}nm available.")

    rows = []
    reporter = ProgressReporter(total=len(SWEEP_B_N_FRAMES), label="Sweep B (single-FOV frames)")
    for n in reporter.wrap(SWEEP_B_N_FRAMES):
        if n > len(all_frame_samples):
            continue
        field, _meta = compute_ffc_field_for_color(
            all_frame_samples[:n], config.frame_width, config.frame_height,
            FFC_SMOOTH_SIGMA_PX, FFC_NORMALIZE_PERCENTILE, FFC_MIN_VALUE,
        )
        median_pct, p95_pct = percent_diff_vs_reference(field, reference_field)
        rows.append({"n_frames": n, "median_pct_diff": median_pct, "p95_pct_diff": p95_pct})

    sweep_b_df = pd.DataFrame(rows)
    sweep_b_df.to_csv(sweep_b_cache, index=False)

sweep_b_df

Plot both convergence curves:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(sweep_a_df["n_fovs"], sweep_a_df["median_pct_diff"], "o-",
        label="Sweep A: N emptiest FOVs (1 frame each)")
ax.plot(sweep_b_df["n_frames"], sweep_b_df["median_pct_diff"], "s-",
        label="Sweep B: 1 FOV, N frames")
ax.axhline(CONVERGENCE_TOLERANCE_PCT, color="r", ls="--",
           label=f"{CONVERGENCE_TOLERANCE_PCT:.1f}% tolerance")

ax.set_xlabel("Sample count (FOVs or frames)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("Median % difference vs. reference field", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"FFC convergence -- round {ROUND_ID}, {COLOR_NM}nm", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
fig.tight_layout()
fig.savefig(cache_dir / "convergence_curves.png", dpi=150)
plt.show()

def first_below_tolerance(df, count_col):
    passing = df[df["median_pct_diff"] <= CONVERGENCE_TOLERANCE_PCT]
    return int(passing[count_col].min()) if len(passing) else None

print(f"Sweep A: smallest N FOVs below {CONVERGENCE_TOLERANCE_PCT}% tolerance: "
      f"{first_below_tolerance(sweep_a_df, 'n_fovs')}")
print(f"Sweep B: smallest N frames below {CONVERGENCE_TOLERANCE_PCT}% tolerance: "
      f"{first_below_tolerance(sweep_b_df, 'n_frames')}")

## 7 -- Conclusion

Fill in after running against a real experiment:

- Smallest N FOVs (sweep A) below tolerance: **TBD**
- Smallest N frames, single FOV (sweep B) below tolerance: **TBD**
- Which strategy converges faster / needs fewer real reads: **TBD**
- Recommended production defaults for `common/config.py`:
  - `ffc_fov_selection_strategy = "TBD"`
  - `ffc_min_samples = TBD`